# Exercise 1 - Product extractor

## Import data

In [3]:
import json
from pathlib import Path

with open(Path("data/products.json"), "r", encoding="utf-8") as f:
    products = json.load(f)

descriptions = [product["description"] for product in products]

print(f"Loaded {len(descriptions)} descriptions")
print(descriptions[0])

Loaded 10 descriptions
The Nike Running Shoes X90 are available for $120.00 in the Footwear section.


## Create pydantic model and agent

In [5]:
from pydantic import BaseModel, Field

class ProductInfo(BaseModel):
    name: str = Field(description="Product name")
    price: float = Field(description="Product price as number")
    category: str = Field(description="Product category")
    in_stock: bool = Field(description="Whether the product is in stock")
    description: str = Field(description="Product Short cleaned product description")
    

In [6]:
from pydantic_ai import Agent

product_extractor_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt=""" 
You extract strcture product information from product descriptions.

Extract these fields:
- name
- price
- category
- in_stock
- description

rules: 
- Only extract information supported by the text.
- price must be numeric only, without currency symbol. 
- in_stock must be true or false.
- description should be a short clenaed summary of product text.
- If a value is missing, make your best grounded inference only if strongly supported by the text. 
"""
)

async def extract_product(descriptions_text: str) -> ProductInfo:
    result = await product_extractor_agent.run(
        descriptions_text,
        output_type=ProductInfo
    )
    return result.output


In [7]:
structured_products = []

for desc in descriptions:
    product = await extract_product(desc)
    structured_products.append(product)

structured_products[:3]

[ProductInfo(name='Nike Running Shoes X90', price=120.0, category='Footwear', in_stock=True, description='Nike Running Shoes X90 available for $120.00 in the Footwear section.'),
 ProductInfo(name='Sony WH-1000XM5 Headphones', price=349.99, category='Electronics', in_stock=False, description='Sony WH-1000XM5 Headphones in Electronics department.'),
 ProductInfo(name='Dyson V12 Vacuum Cleaner', price=599.99, category='Home & Garden', in_stock=False, description='Currently out of stock but available for pre-order.')]

In [8]:
from pydantic import BaseModel, Field, field_validator

class ProductInfo(BaseModel):
    name: str = Field(min_length=1)
    price: float
    category: str = Field(min_length=1)
    in_stock: bool
    description: str = Field(min_length=5)

    @field_validator("price")
    @classmethod
    def validate_pricce(cls, v):
        if v < 0:
            raise ValueError("price must be non-negative")
        return v
    
    @field_validator("category")
    @classmethod
    def validate_category(cls, v):
        allowed_categories = {
            "electronics",
            "clothing",
            "home",
            "beauty",
            "sports",
            "books",
            "toys",
            "food",
            "other",
        }
        if v.lower() not in allowed_categories:
            raise ValueError(f"invalid category: {v}")
        return v

In [10]:
validated_products = []
failed_products = []

for i, desc in enumerate(descriptions):
    try:
        result = await product_extractor_agent.run(
            desc,
            output_type=ProductInfo
        )
        validated_products.append(result.output)
    except Exception as e:
        failed_products.append({
            "index": i,
            "description": desc,
            "error": str(e)
        })
print ("Valid:", len(validated_products))
print ("Failed:", len(failed_products))

Valid: 6
Failed: 4
